# Remote Work and Wage Inequality: Correcting Bias in Regression with Generated Binary Labels

In [ ]:
# Run this cell first on Google Colab to install the bias-correction package.
%pip install "ValidMLInference>=1.4.0"

This notebook estimates the association between working from home and salaries using real-world job postings data [(Hansen et al., 2023)](https://dx.doi.org/10.2139/ssrn.4380734). It illustrates how the functions `ols_bca`, `ols_bcm` and `one_step` can be used to correct bias from regressing on AI/ML-generated labels. The notebook reproduces results from Table 1 of [Battaglia, Christensen, Hansen & Sacher (2024)](https://arxiv.org/abs/2402.15585).

In [ ]:
from ValidMLInference import ols, ols_bcm, remote_work_data
import numpy as np

SD_data = remote_work_data()
SD_data['salary'] = np.log(SD_data['salary'])

In [ ]:
from ValidMLInference import ols, ols_bcm, remote_work_data
import numpy as np

# transform salary to log scale
SD_data = remote_work_data()
SD_data['salary'] = np.log(SD_data['salary'])

# illustrative regression (with occupation and full-time/part-time effects)
formula = "salary ~ wfh_wham + C(soc_2021_2) + C(employment_type_name)"

# two-step approach
mod_2step = ols(formula = formula, data = SD_data)
print(mod_2step.summary().loc[["Intercept", "wfh_wham"]])

# bias-corrected two-step approach
mod_2step_bc = ols_bcm (formula = formula, generated_var = "wfh_wham", data = SD_data, fpr = 0.009, m = 1000)
print(mod_2step_bc.summary().loc[["Intercept", "wfh_wham"]])

## Data

The package contains a subset of [a larger dataset](https://wfhmap.com/) regarding work from home. The sample consists of 16,315 job postings for 2022 and 2023 with “San Diego, CA” recorded as the city and “72” recorded as the NAICS2 industry code of the advertising firm. 

The data set contains the following entries:
1. `city_name` 
2. `naics_2022_2` - an industry code 
3. `salary` 
4. `wfh_wham` - ML-generated indicator of whether the job offers work from home using fine-tuned DistilBERT as in [(Hansen et al., 2023)](https://dx.doi.org/10.2139/ssrn.4380734)
5. `soc_2021_2` - Bureau of Labor Statistics Standard Occupational Classification code
6. `employment_type_name` - indicates whether the position is full-time or part-time 

In [ ]:
SD_data = remote_work_data()
SD_data.head(5)

For purpose of this estimation, we also log-transform the salary data. 

In [ ]:
SD_data['salary'] = np.log(SD_data['salary'])

## Estimating the false-positive rate

The variable `wfh_wham` describing whether the job posting offers remote work is not manually collected, but is imputed via ML methods using fine-tuned DistilBERT as in [(Hansen et al., 2023)](https://dx.doi.org/10.2139/ssrn.4380734). This classifier has over 99% test accuracy in the full sample. Nevertheless, as [Battaglia, Christensen, Hansen & Sacher (2024)](https://arxiv.org/abs/2402.15585) document, even high-performance classifiers can lead to large biases in OLS estimates.

The bias correction methods `ols_bca` and `ols_bcm` require estimates of the classifier's false-positive rate.

We estimate the false positive rate manually. To do so, we took a random sample of size 1000 postings. Of these, 26 had `wfh_wham = 1`. Based on reading these 26 postings, 9 appeared to be misclassified. This means the estimated false-positive rate is 0.009. Accordingly, we will implement `ols_bcm` with `fpr = 0.009` (the estimated false-positive rate) and `m = 1000` (the sample size used to estimate the false-positive rate).

## Results

We first present results for a simple regression of log salary onto the remote work indicator. We then consider a second specification with fixed effects.

We compare standard OLS esitmates and confidence intervals with estimates and confidence intervals using `ols_bcm` which performs a direct bias correction and computes bias corrected CIs, and `one_step` which performs maximum likelihood estimation treating the true labels as latent.

### Without fixed effects

We first present OLS estimates:

In [ ]:
from ValidMLInference import ols
res = ols(formula = "salary ~ wfh_wham", data=SD_data, intercept = True)
print(res.summary())

Now using the multiplicative bias correction, with bias corrected CIs:

In [ ]:
res = ols_bcm(formula= "salary ~ wfh_wham", data=SD_data, fpr = 0.009, m = 1000, intercept=True)
print(res.summary())

The second correction provided in the package is maximum likelihood.  The simplest function for this is `one_step` which can take any JAX-compatible PDF function as an error distribution.  For added flexibility, there is also a `one_step_gaussian_mixture` function that models the error term as a Gaussian mixutre.  Below we show the residuals from the `ols` call in two-step estimation have a complex structure.  We therefore use a three-component Gaussian mixture for the error distribution.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Get the fitted values and calculate residuals manually

y = SD_data['salary'].values
X = np.column_stack([np.ones(len(SD_data)), SD_data['wfh_wham'].values])

# Get the fitted values using the coefficients from our result
fitted_values = X @ res.coef

# Calculate residuals
residuals = y - fitted_values

# Now plot the distribution of residuals
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Histogram of residuals
ax1.hist(residuals, bins=50, alpha=0.7, edgecolor='black')
ax1.set_xlabel('Residuals')
ax1.set_ylabel('Frequency')
ax1.set_title('Distribution of Residuals')
ax1.grid(True, alpha=0.3)

# Q-Q plot to check normality
from scipy import stats
stats.probplot(residuals, dist="norm", plot=ax2)
ax2.set_title('Q-Q Plot of Residuals')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print some summary statistics
print(f"Mean of residuals: {residuals.mean():.6f}")
print(f"Standard deviation: {residuals.std():.6f}")
print(f"Skewness: {stats.skew(residuals):.6f}")
print(f"Kurtosis: {stats.kurtosis(residuals):.6f}")

The implementation of one-step estimation is the following:

In [ ]:
res = one_step_gaussian_mixture(formula = "salary ~ wfh_wham", data = SD_data, generated_var = "wfh_wham", k = 3, nguess = 30, maxiter=300, seed = 123) 
print(res.summary())

### With fixed effects

We repeat the above now with fixed effects, which are easily generated for the categorical variables `soc_2021_2` and `employment_type_name`.

First using OLS:

In [ ]:
res = ols(formula = "salary ~ wfh_wham + C(soc_2021_2) + C(employment_type_name)", data = SD_data, intercept=True)
summary = res.summary()
rows = ["Intercept", "wfh_wham"]
print(summary.loc[rows])

Now using the multiplicative bias correction:

In [ ]:
res = ols_bcm(formula = "salary ~ wfh_wham + C(soc_2021_2) + C(employment_type_name)", generated_var = "wfh_wham", data = SD_data, fpr = 0.009, m=1000)
summary = res.summary()
print(summary.loc[rows])

Comparing these results with the OLS results above, we see that the bias corrected CI for the slope coefficient lies to the right of the OLS point estimate.

Finally, using maximum likelihood, again with a three-component Gaussian mixture distribution for the error term:

In [ ]:
res = one_step_gaussian_mixture(formula = "salary ~ wfh_wham + C(soc_2021_2) + C(employment_type_name)", data = SD_data, generated_var = "wfh_wham", k = 3, nguess = 30, maxiter=300) 
summary = res.summary()
print(summary.loc[rows])